# 29. The encoder's smoothing constant

**One variable against ledger row 38** (`xgb_te`, CV 0.967099): the encoder's smoothing
constant. Same learner, same folds, same seed, same budget, same inner split count.
Only `SMOOTH` moves.

## Why this is the last untested thing here

Target encoding is the largest gain in this competition, +0.003312 over row 9. It
arrived on 2026-08-11 with `SMOOTH = 10.0` and `N_INNER = 5` written into cell 1 of
`13_target_encoding.ipynb`, chosen once, and **every model since has inherited both
without either ever being varied**: the five LightGBM seeds, the five CatBoost seeds,
both neural models and now the five XGBoost seeds. Twenty-two of the twenty-nine
members of row 44 depend on a constant nobody has tested.

`SMOOTH` is swept here. `N_INNER` is a second variable and would be a second run.

## The prior, and why it is low

The smoothed mean is `(sum + prior * SMOOTH) / (count + SMOOTH)`. At `SMOOTH = 10`
against roughly 500 rows per level on the float columns, the shrinkage toward the prior
is about 2 percent, which is nearly nothing. The constant is close to inert by
construction on the high-cardinality columns, and on the three categoricals the levels
hold tens of thousands of rows, where it is inert twice over.

So the expectation is a flat curve, and the grid runs to 100 to make sure the flat part
is actually being left before the curve is called flat.

**That prior deserves less confidence than it reads with.** `num_leaves` was predicted
flat and was flat, but XGBoost was predicted to return little and returned the best
model in the repo, and both predictions were written the same day with the same
confidence.

## The gate, pre-registered, and it is deliberately not `23`'s

`23_lgbm_leaves.ipynb` set its floor at the fold spread, 0.00045. **That floor was too
strict and this run does not reuse it.** XGBoost's single-model gain over LightGBM was
+0.000316, which the seed sweep in rows 40 to 43 later confirmed at 35 standard errors,
and a 0.00045 floor would have called it a null. Recording that because the floor was
mine and it was wrong.

The test is the one `NOTES.md` settled on, the spread of the per-fold difference and
how many folds it wins, against the `SMOOTH = 10` arm from this same kernel.

| verdict | condition |
|---|---|
| `carry to a second seed` | wins >= 4/5, mean > 2 x paired sd, and mean >= +0.00010 |
| `parity` | paired-significant but under +0.00010 |
| `null` | anything else |

The +0.00010 floor sits just below **+0.000132**, which is CatBoost's margin in row 26
and the smallest single-model difference this repo has since confirmed real through a
seed sweep. No arm is called an improvement here on one seed, per rows 26 and 38.

## The encoder is unchanged, only its parameter moves

`_stats` reads `SMOOTH` from the module globals, so each arm rebinds the constant and
calls the same function. The encoder source is untouched and still fingerprints
`0642e41750ef8bab`, which is what makes this one variable rather than a rewrite.

Unlike `23`, the encoder cannot be shared across arms: `SMOOTH` is an input to it, so
each arm rebuilds it. That is 25 encoder builds rather than 5, and at about 2 seconds
each it is not the cost that matters.

In [ ]:
# One flag. The sweep always runs top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5

# The knob under test. 10.0 is 13's value and therefore row 38's, so that arm is the
# reproduction check.
SMOOTHS = [1.0, 5.0, 10.0, 25.0, 100.0]
BASE_SMOOTH = 10.0
SMOOTH = BASE_SMOOTH   # rebound per arm below; the encoder reads it from globals

# Row 38's configuration, held.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0
MAX_DEPTH = 6
N_JOBS = -1

BASELINE_NAME = "xgb_te"
BASELINE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

GATE_FLOOR = 1.0e-04

print(f"SMOKE = {SMOKE}   smoothing {SMOOTHS}   base {BASE_SMOOTH}")

## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up in NOTES.md: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones in `NOTES.md`. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench, determinism, and a check that the constant is live

The third check matters more than usual here. If `SMOOTH` were somehow not reaching the
encoder, every arm would be identical, the sweep would report a perfectly flat curve,
and a flat curve is exactly the result the header predicts. That coincidence would be
indistinguishable from success, so it is tested rather than assumed.

In [ ]:
import xgboost as xgb


def make(n_est):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )


def fit_arm(Xtr, ytr, Xva, n_est, Xte=None):
    m = make(n_est)
    t0 = time.time()
    m.fit(Xtr, ytr)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if Xte is not None else None
    return p, p_te, secs


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "29_encoder_smooth.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, smoothing={SMOOTHS} ===")

tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]

# Is the constant live? Build the same fold at two very different values and compare
# the encoded columns directly, before any model sees them.
SMOOTH = BASE_SMOOTH
_t0 = time.time()
Xa, _, _ = build(X, y, tr0, va0)
ENC_SECS = time.time() - _t0
SMOOTH = 100.0
Xb, _, _ = build(X, y, tr0, va0)
SMOOTH = BASE_SMOOTH

te_cols = [c for c in Xa.columns if c.startswith("te_")]
moved = float(np.abs(Xa[te_cols].to_numpy() - Xb[te_cols].to_numpy()).max())
SMOOTH_LIVE = moved > 0
print(f"encoder, one fold: {hhmm(ENC_SECS)}   {Xa.shape[1]} features")
print(f"SMOOTH 10 vs 100, largest change in an encoded column: {moved:.3e}  "
      f"{'live' if SMOOTH_LIVE else 'NOT LIVE - the sweep would be five copies'}")
del Xb
gc.collect()

Xtr0, Xva0, _ = build(X, y, tr0, va0)
pa, _, sa = fit_arm(Xtr0, y[tr0], Xva0, BENCH_EST)
pb, _, _ = fit_arm(Xtr0, y[tr0], Xva0, BENCH_EST)
delta = float(np.abs(pa - pb).max())
DETERMINISTIC = delta == 0.0
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")

per_tree = sa / BENCH_EST
total = 5 * len(SMOOTHS) * (ENC_SECS + per_tree * N_EST)
print()
print(f"projection, {N_EST} trees, 5 folds x {len(SMOOTHS)} arms: {hhmm(total)}")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"smooth live {SMOOTH_LIVE}, projected {hhmm(total)}")

del Xa, Xtr0, Xva0, pa, pb
gc.collect()

## Stage 4. The sweep

Each arm rebinds `SMOOTH`, rebuilds the encoder inside every fold, and trains the same
XGBoost configuration. The fold partition, the inner split and the model seed are
identical across arms.

In [ ]:
oof = {s: np.zeros(len(train)) for s in SMOOTHS}
test_pred = {s: np.zeros(len(test)) for s in SMOOTHS}
per_fold = {s: [] for s in SMOOTHS}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    for s in SMOOTHS:
        SMOOTH = s
        Xtr, Xva, Xte = build(X, y, tr, va, X_test)
        p, p_te, secs = fit_arm(Xtr, y[tr], Xva, N_EST, Xte)
        oof[s][va] = p
        test_pred[s] += p_te / 5
        per_fold[s].append(float(roc_auc_score(y[va], p)))
        note(f"  fold {f} smooth {s:>6}: {per_fold[s][-1]:.6f}  ({hhmm(secs)})")
        del Xtr, Xva, Xte
        gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")
SMOOTH = BASE_SMOOTH

cv = {s: float(np.mean(per_fold[s])) for s in SMOOTHS}
sd = {s: float(np.std(per_fold[s])) for s in SMOOTHS}

print()
print(f"{'smooth':>8} {'CV':>10} {'fold sd':>10}")
for s in SMOOTHS:
    star = "  <- 13's value, and row 38's" if s == BASE_SMOOTH else ""
    print(f"{s:>8} {cv[s]:>10.6f} {sd[s]:>10.6f}{star}")
note("sweep done, " + ", ".join(f"{s}:{cv[s]:.6f}" for s in SMOOTHS))

In [ ]:
repro = cv[BASE_SMOOTH] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4
print(f"arm SMOOTH={BASE_SMOOTH}: {cv[BASE_SMOOTH]:.6f}")
print(f"ledger row 38       : {BASELINE_CV:.6f}")
print(f"difference          : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print("SMOKE: subsampled, so this is EXPECTED to be far out and is not a check.")

base = np.array(per_fold[BASE_SMOOTH])
rows = []
for s in SMOOTHS:
    if s == BASE_SMOOTH:
        continue
    d = np.array(per_fold[s]) - base
    rows.append((s, d.mean(), d.std(ddof=1), int((d > 0).sum()), d))

print()
print(f"{'smooth':>8} {'paired mean':>13} {'paired sd':>11} {'wins':>7} {'mean/sd':>9}")
for s, m, sdv, w, _ in rows:
    print(f"{s:>8} {m:>+13.6f} {sdv:>11.6f} {w:>5}/5 "
          f"{(m / sdv if sdv else float('nan')):>9.1f}")
print()
for s, m, sdv, w, d in rows:
    print(f"  {s:>6}: per-fold {np.round(d, 6).tolist()}")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a leak check failed"
elif not ENCODER_MATCH:
    blocked = "the encoder does not match 13"
elif not DETERMINISTIC:
    blocked = "the configuration is not reproducible"
elif not SMOOTH_LIVE:
    blocked = "SMOOTH is not reaching the encoder, so these are not five arms"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed"
elif not SMOKE and not REPRODUCED:
    blocked = f"the SMOOTH={BASE_SMOOTH} arm missed row 38 by {repro:+.2e}"

print()
if blocked:
    print(f"VERDICT: blocked, {blocked}")
elif SMOKE:
    print("SMOKE: no verdict, subsampled rows cannot resolve differences this small.")
else:
    best = max(rows, key=lambda r: r[1])
    s, m, sdv, w, _ = best
    sig = w >= 4 and sdv > 0 and m > 2 * sdv
    if sig and m >= GATE_FLOOR:
        print(f"VERDICT: carry to a second seed. SMOOTH={s} clears the bar at {m:+.6f},")
        print("  and one seed does not make an improvement in this repo (rows 26, 38).")
    elif sig:
        print(f"VERDICT: parity. SMOOTH={s} is paired-significant at {m:+.6f} but under")
        print(f"  the {GATE_FLOOR:.5f} floor. Logged as parity, not as an improvement.")
    else:
        print("VERDICT: null. The curve is flat and 13's SMOOTH=10 stands. The constant")
        print("  is inert on this data, as the header predicted from the arithmetic.")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for s in SMOOTHS:
    tag = str(s).replace(".", "p")
    np.save(OUT / f"{pre}xgb_smooth{tag}_oof.npy", oof[s])
    np.save(OUT / f"{pre}xgb_smooth{tag}_test.npy", test_pred[s])
print(f"wrote {pre}xgb_smooth<v>_oof.npy and _test.npy for {SMOOTHS}")
print()
print("ledger lines, one per arm:")
for s in SMOOTHS:
    print(f"  xgb_smooth{str(s):<6}  cv_mean {cv[s]:.6f}  cv_std {sd[s]:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
      f"smooth live {'yes' if SMOOTH_LIVE else 'NO'}")